# Experiment 11: town-segmented models

Train a global rental model and separate AutoGluon models for each sufficiently large `TOWN`. At prediction time, each row is routed to its town model; sparse or unseen towns fall back to the global model. Validation is chronological. HPO is deliberately disabled for this first comparison.

In [1]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from scipy.spatial import cKDTree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'), Path('/content/drive/MyDrive/KDDM-project')]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None: raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets

TARGET, GROUP_COLUMNS, LAG_MONTHS = 'MONTHLY_RENT', ['TOWN', 'FLAT_TYPE'], 24
RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
TIME_LIMIT_PER_MODEL = int(os.environ.get('TOWN_SEGMENT_TIME_LIMIT', '300'))
MIN_SEGMENT_TRAIN_ROWS = int(os.environ.get('MIN_SEGMENT_TRAIN_ROWS', '2000'))
compact_features = ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km']
MODEL_FEATURES = compact_features + ['monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name', 'nearest_tourist_attraction_name', 'nearest_monument_name']

_, datasets = load_datasets(project_root / 'data')
raw_train = datasets['train'].copy()
raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
periods = np.sort(raw_train['period'].unique())
validation_cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw_train.loc[raw_train.period < validation_cutoff].copy()
raw_valid = raw_train.loc[raw_train.period >= validation_cutoff].copy()
print(f'Chronological cutoff: {validation_cutoff}; fit={len(raw_fit):,}, validation={len(raw_valid):,}; min town size={MIN_SEGMENT_TRAIN_ROWS:,}; HPO disabled')


Chronological cutoff: 2024-05; fit=117,988, validation=32,012; min town size=2,000; HPO disabled


In [2]:
class TargetFeatureTransformer:
    def __init__(self, lag_months=LAG_MONTHS): self.lag_months = lag_months
    def fit(self, history):
        self.global_mean_ = history[TARGET].mean()
        self.group_mean_ = history.groupby(GROUP_COLUMNS)[TARGET].mean()
        self.monthly_mean_ = history.groupby('period')[TARGET].mean()
        self.group_period_ = history.groupby(GROUP_COLUMNS + ['period'], as_index=False)[TARGET].mean().rename(columns={'period':'source_period', TARGET:'town_flat_type_target_lag'})
        return self
    def transform(self, history):
        result = history[GROUP_COLUMNS + ['period']].copy()
        result['monthly_target_lag'] = (result.period - self.lag_months).map(self.monthly_mean_).fillna(self.global_mean_)
        result['source_period'] = result.period - self.lag_months
        result = result.merge(self.group_period_, on=GROUP_COLUMNS + ['source_period'], how='left').drop(columns='source_period')
        group_mean = pd.Series(result.set_index(GROUP_COLUMNS).index.map(self.group_mean_), index=result.index)
        result['town_flat_type_target_mean'] = group_mean.fillna(self.global_mean_)
        result['town_flat_type_target_lag'] = result.town_flat_type_target_lag.fillna(result.town_flat_type_target_mean)
        return result[['monthly_target_lag', 'town_flat_type_target_mean', 'town_flat_type_target_lag']]
    def fit_transform_train(self, history):
        self.fit(history)
        grouped = history.groupby(GROUP_COLUMNS + ['period'], as_index=False)[TARGET].agg(['sum','count']).reset_index().sort_values(GROUP_COLUMNS + ['period'])
        grouped['prior_sum'] = grouped.groupby(GROUP_COLUMNS)['sum'].cumsum() - grouped['sum']
        grouped['prior_count'] = grouped.groupby(GROUP_COLUMNS)['count'].cumsum() - grouped['count']
        grouped['town_flat_type_target_mean'] = grouped.prior_sum / grouped.prior_count.replace(0, np.nan)
        global_monthly = history.groupby('period')[TARGET].agg(['sum','count']).sort_index()
        global_prior = (global_monthly['sum'].cumsum()-global_monthly['sum']) / (global_monthly['count'].cumsum()-global_monthly['count']).replace(0, np.nan)
        result = history[GROUP_COLUMNS + ['period']].merge(grouped[GROUP_COLUMNS + ['period','town_flat_type_target_mean']], on=GROUP_COLUMNS + ['period'], how='left')
        result['town_flat_type_target_mean'] = result.town_flat_type_target_mean.fillna(result.period.map(global_prior))
        result['source_period'] = result.period - self.lag_months
        result = result.merge(self.group_period_, on=GROUP_COLUMNS + ['source_period'], how='left').drop(columns='source_period')
        result['town_flat_type_target_lag'] = result.town_flat_type_target_lag.fillna(result.town_flat_type_target_mean)
        result['monthly_target_lag'] = (result.period-self.lag_months).map(self.monthly_mean_).fillna(result.period.map(global_prior))
        return result[['monthly_target_lag', 'town_flat_type_target_mean', 'town_flat_type_target_lag']]

def preprocess_partition(preprocessor, frame, fit=False):
    values = frame.drop(columns=['period', TARGET], errors='ignore')
    return preprocessor.fit_transform(values) if fit else preprocessor.transform(values)
def history_from(frame, base):
    result = base[GROUP_COLUMNS].reset_index(drop=True).copy(); result['period'] = frame.period.to_numpy(); result[TARGET] = frame[TARGET].to_numpy(); return result
def nearest_mrt_name(base):
    stations = datasets['mrt'][['NAME','LATITUDE','LONGITUDE']].dropna()
    _, index = cKDTree(stations[['LATITUDE','LONGITUDE']].to_numpy()*[111.0,110.96]).query(base[['LATITUDE','LONGITUDE']].to_numpy()*[111.0,110.96])
    return pd.Series(stations.NAME.astype(object).to_numpy()[index], name='nearest_mrt_name')
def nearest_geojson_name(base, filename, name_field, feature_name):
    with (project_root / 'data' / 'possible_additional_features' / filename).open() as handle: data = json.load(handle)
    points = pd.DataFrame([{'longitude':f['geometry']['coordinates'][0], 'latitude':f['geometry']['coordinates'][1], feature_name:f['properties'][name_field]} for f in data['features'] if f.get('geometry',{}).get('type') == 'Point']).dropna()
    _, index = cKDTree(points[['longitude','latitude']].to_numpy()*[110.96,111.0]).query(base[['LONGITUDE','LATITUDE']].to_numpy()*[110.96,111.0])
    return pd.Series(points[feature_name].astype(object).to_numpy()[index], name=feature_name)


In [3]:
# Fit every transformer on raw_fit only; validation is transform-only.
preprocessor = RentalPreprocessor(datasets)
fit_base = preprocess_partition(preprocessor, raw_fit, fit=True)
valid_base = preprocess_partition(preprocessor, raw_valid)
target_transformer = TargetFeatureTransformer()
fit_target = target_transformer.fit_transform_train(history_from(raw_fit, fit_base))
valid_target = target_transformer.transform(history_from(raw_valid, valid_base))
def make_model_frame(base, target_features):
    return pd.concat([base[compact_features].reset_index(drop=True), target_features.reset_index(drop=True), nearest_mrt_name(base).reset_index(drop=True), nearest_geojson_name(base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name').reset_index(drop=True), nearest_geojson_name(base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name').reset_index(drop=True)], axis=1)[MODEL_FEATURES]
fit_x, valid_x = make_model_frame(fit_base, fit_target), make_model_frame(valid_base, valid_target)
fit_y, valid_y = raw_fit[TARGET].astype(float).reset_index(drop=True), raw_valid[TARGET].astype(float).reset_index(drop=True)
assert len(fit_x) == len(fit_y) and len(valid_x) == len(valid_y)


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [4]:
coverage = pd.DataFrame({'fit_rows': fit_x.groupby('TOWN').size(), 'validation_rows': valid_x.groupby('TOWN').size()}).fillna(0).astype(int)
coverage['train_segment_model'] = coverage.fit_rows >= MIN_SEGMENT_TRAIN_ROWS
display(coverage.sort_values('fit_rows', ascending=False))
eligible_towns = coverage.index[coverage.train_segment_model].tolist()
print('Eligible town models:', eligible_towns)


,fit_rows,validation_rows,train_segment_model
TOWN,,,
jurong west,8235,2247,True
tampines,7893,2049,True
sengkang,7653,2120,True
bedok,7011,1896,True
ang mo kio,6513,1770,True
bukit merah,6413,1857,True
yishun,6258,1655,True
woodlands,5993,1584,True
hougang,5620,1495,True


Eligible town models: ['ang mo kio', 'bedok', 'bishan', 'bukit batok', 'bukit merah', 'bukit panjang', 'choa chu kang', 'clementi', 'geylang', 'hougang', 'jurong east', 'jurong west', 'kallang whampoa', 'pasir ris', 'punggol', 'queenstown', 'sembawang', 'sengkang', 'serangoon', 'tampines', 'toa payoh', 'woodlands', 'yishun']


In [5]:
def fit_predictor(segment_x, segment_y, name):
    return TabularPredictor(
        label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_exp_11_town' / RUN_ID / name),
    ).fit(
        train_data=segment_x.assign(**{TARGET: segment_y.to_numpy()}),
        hyperparameters={'GBM': {}, 'CAT': {}}, fit_weighted_ensemble=True,
        presets=['medium_quality', 'ignore_text'], time_limit=TIME_LIMIT_PER_MODEL, verbosity=2,
    )
def metrics(name, actual, prediction):
    return {'model':name, 'RMSE':mean_squared_error(actual,prediction)**0.5, 'MAE':mean_absolute_error(actual,prediction), 'R2':r2_score(actual,prediction)}

global_predictor = fit_predictor(fit_x, fit_y, 'global')
global_prediction = global_predictor.predict(valid_x).astype(float).to_numpy()
segmented_prediction = global_prediction.copy()
routing = pd.Series('global_fallback', index=valid_x.index, dtype=object)
segment_predictors = {}
for town in eligible_towns:
    train_mask, valid_mask = fit_x.TOWN.eq(town), valid_x.TOWN.eq(town)
    if not valid_mask.any(): continue
    predictor = fit_predictor(fit_x.loc[train_mask], fit_y.loc[train_mask], f'town_{town}')
    segmented_prediction[valid_mask] = predictor.predict(valid_x.loc[valid_mask]).astype(float).to_numpy()
    routing.loc[valid_mask] = f'town_{town}'
    segment_predictors[town] = predictor

comparison = pd.DataFrame([metrics('Global AutoGluon', valid_y, global_prediction), metrics('Town-routed AutoGluon', valid_y, segmented_prediction)])
display(comparison.sort_values('RMSE'))
segment_metrics = pd.DataFrame([metrics(town, valid_y.loc[valid_x.TOWN.eq(town)], segmented_prediction[valid_x.TOWN.eq(town)]) for town in sorted(valid_x.TOWN.dropna().unique())])
display(segment_metrics.sort_values('RMSE'))
display(routing.value_counts().rename_axis('route').reset_index(name='validation_rows'))
print('HPO is disabled: every global/town model uses the same fixed AutoGluon configuration.')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.62 GB / 16.00 GB (28.9%)
Disk Space Avail:   180.30 GB / 460.38 GB (39.2%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_11_town/20260927_172210/global"
Train Data Rows:    117988
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess t

,model,RMSE,MAE,R2
0,Global AutoGluon,468.271759,340.986810,0.491986
1,Town-routed AutoGluon,475.845617,350.585699,0.475420


,model,RMSE,MAE,R2
17,punggol,377.411084,250.390073,0.333408
20,sengkang,377.448262,262.454717,0.297892
19,sembawang,411.917341,305.525737,0.371875
5,bukit panjang,413.034638,295.246621,0.357551
3,bukit batok,441.439260,328.814717,0.498051
8,choa chu kang,446.199203,319.525203,0.233305
15,marine parade,447.808127,327.213895,0.449027
0,ang mo kio,448.757183,332.806902,0.400090
11,hougang,451.833274,336.227198,0.387494
22,tampines,458.977422,335.434303,0.375809


,route,validation_rows
0,town_jurong west,2247
1,town_sengkang,2120
2,town_tampines,2049
3,town_bedok,1896
4,town_bukit merah,1857
5,town_ang mo kio,1770
6,town_yishun,1655
7,town_woodlands,1584
8,town_punggol,1498
9,town_hougang,1495


HPO is disabled: every global/town model uses the same fixed AutoGluon configuration.
